# 01 Webcam-Based PM2.5 Dataset Pipeline

This notebook implements the complete workflow for constructing a webcam-based PM2.5 dataset, including:

- Webcam image loading and ROI selection
- Image feature extraction
- PM2.5, ERA5, and ARPA meteorological data processing
- Multi-source dataset merging and cleaning

The final output is an analysis-ready dataset for subsequent air quality modeling and analysis.


## Setup

In [ ]:
import importlib
import json
import sys

from pathlib import Path

import pandas as pd
from IPython.display import display

## Cell 1 — Load Collected Webcam Images
Load all collected webcam images and initialize the project paths.

In [ ]:
# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Define image directory
if IN_COLAB:
    drive.mount('/content/drive')

    # Google Drive folder
    IMAGE_DIR = Path('/content/drive/MyDrive/webcam_images')

else:
    # notebooks/ -> project root

    PROJECT_ROOT = Path.cwd().parent

    if str(PROJECT_ROOT) not in sys.path:

        sys.path.append(str(PROJECT_ROOT))

    IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "images"

image_files = list(IMAGE_DIR.glob("*.jpg"))

print("Running in Colab:", IN_COLAB)

if not IN_COLAB:
    print("Project root:", PROJECT_ROOT.name)

print(
    "Image directory:",
    IMAGE_DIR.relative_to(PROJECT_ROOT)
    if not IN_COLAB
    else IMAGE_DIR
)

print("Number of images:", len(image_files))

Running in Colab: False
Project root: webcam-pm25-toolbox
Image directory: data/raw/images
Number of images: 220


## Cell 2 — ROI Extraction
Interactively select the region of interest (ROI) used for feature extraction.

In [6]:
import src.roi_viewer as roi_viewer

importlib.reload(roi_viewer)

roi_controls = roi_viewer.build_roi_viewer(IMAGE_DIR)

Output()

In [4]:
selected_roi = {
    "top": roi_controls["top"].value,
    "left": roi_controls["left"].value,
    "height": roi_controls["height"].value,
    "width": roi_controls["width"].value
}

ROI_JSON = PROJECT_ROOT / "config" / "roi.json"
ROI_JSON.parent.mkdir(parents=True, exist_ok=True)

with open(ROI_JSON, "w") as f:
    json.dump(selected_roi, f, indent=4)

print("Selected ROI:")
print(f"top = {selected_roi['top']}")
print(f"left = {selected_roi['left']}")
print(f"height = {selected_roi['height']}")
print(f"width = {selected_roi['width']}")
print(f"mode = {roi_controls['mode'].value}")

print("\nROI saved to:")
print(ROI_JSON.relative_to(PROJECT_ROOT))

Selected ROI:
top = 160
left = 116
height = 380
width = 515
mode = RGB

ROI saved to:
config/roi.json


## Cell 3 — Image Feature Extraction
Extract RGB, saturation, contrast, and B/R ratio features from all webcam images.

In [8]:
import src.image_features as image_features

importlib.reload(image_features)

IMAGE_FEATURES_CSV = PROJECT_ROOT / "data" / "interim" / "image_features.csv"

rows, skipped_files = image_features.extract_image_features(
    image_dir=PROJECT_ROOT / "data" / "raw" / "images",
    output_csv=IMAGE_FEATURES_CSV,
    roi=selected_roi
)

df = pd.read_csv(IMAGE_FEATURES_CSV)

df["image_path"] = df["image_path"].apply(
    lambda x: str(Path(x).relative_to(PROJECT_ROOT))
)

df.to_csv(IMAGE_FEATURES_CSV, index=False)

print(f"Image features extracted: {len(rows)} rows")
print("Saved CSV:", IMAGE_FEATURES_CSV.relative_to(PROJECT_ROOT))
print(f"Skipped files: {len(skipped_files)}")

df.head()

Image features extracted: 220 rows
Saved CSV: data/interim/image_features.csv
Skipped files: 0


,datetime,R_roi,G_roi,B_roi,S_mean,B_R_ratio,contrast,image_path
0,2026-03-01 03:00:00,86.927317,103.467195,122.422105,0.290022,1.408327,19.237125,data/raw/images/20260301-0400.jpg
1,2026-03-01 04:00:00,86.576546,103.945815,123.102785,0.295734,1.421895,14.599862,data/raw/images/20260301-0500.jpg
2,2026-03-01 05:00:00,88.301727,103.318487,121.760818,0.273552,1.378918,13.725783,data/raw/images/20260301-0600.jpg
3,2026-03-01 06:00:00,97.634921,127.311369,157.803475,0.381719,1.616261,17.681084,data/raw/images/20260301-0700.jpg
4,2026-03-01 07:00:00,112.163654,144.063168,172.775825,0.352873,1.540390,21.162996,data/raw/images/20260301-0800.jpg


## Cell 4 — Download PM2.5 data
Download hourly PM2.5 observations from the EEA API and save them as a CSV file.

In [9]:
import src.eea_pm25_download as pm25

importlib.reload(pm25)

PM25_CSV = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "PM25_MI_hourly.csv"
)

PM25_TEMP_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pm25_temp"
)

pm25_df = pm25.download_pm25_data(
    api_start="2026-03-01T00:00:00Z",
    api_end="2026-03-12T23:00:00Z",
    station_prefix="IT/SPO.IT0477A_6001_BETA",
    temp_dir=PM25_TEMP_DIR,
    output_file=PM25_CSV,
    remove_temp=True
)

print(
    "Saved CSV:",
    PM25_CSV.relative_to(PROJECT_ROOT)
)
print(f"Rows: {len(pm25_df)}")

pm25_df.head()

Saved CSV: data/interim/PM25_MI_hourly.csv
Rows: 288


,Samplingpoint,Pollutant,Start,End,Value,Unit,AggType,Validity,Verification,ResultTime,DataCapture,FkObservationLog
0,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 00:00:00,2026-03-01 01:00:00,36.062183000000000000,ug.m-3,hour,3,3,2026-03-01 01:00:00,None,9e664d80-8d0a-481e-82ca-112dac15e708
1,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 01:00:00,2026-03-01 02:00:00,25.287216000000000000,ug.m-3,hour,3,3,2026-03-01 02:00:00,None,1499e592-3c02-422b-8646-9dfc91309a4e
2,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 02:00:00,2026-03-01 03:00:00,26.743391000000000000,ug.m-3,hour,3,3,2026-03-01 03:00:00,None,ce0352c6-6450-4676-acd5-03581314f3f2
3,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 03:00:00,2026-03-01 04:00:00,30.829412000000000000,ug.m-3,hour,3,3,2026-03-01 04:00:00,None,d7edb6da-fdba-4913-b409-5fc4b3058c28
4,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 04:00:00,2026-03-01 05:00:00,33.409030000000000000,ug.m-3,hour,3,3,2026-03-01 05:00:00,None,c481d8a8-c512-4069-a081-8057f828c6ac


## Cell 5 — Download and Merge ERA5 data
Download ERA5 meteorological variables and merge single-level and pressure-level data.

In [ ]:
import src.era5_download as era5

importlib.reload(era5)

ERA5_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "era5"

ERA5_CSV = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "era5_all_merged.csv"
)

# Run ERA5 download + processing
era5_df = era5.download_era5_data(
    lat=45.4642,
    lon=9.1900,
    start_date="2026-03-01",
    end_date="2026-03-12",
    work_dir=ERA5_RAW_DIR,
    output_file=ERA5_CSV
)

print(
    "ERA5 raw files saved to:",
    ERA5_RAW_DIR.relative_to(PROJECT_ROOT)
)

print(
    "Merged ERA5 CSV saved to:",
    ERA5_CSV.relative_to(PROJECT_ROOT)
)


display(era5_df.head())

2026-05-18 11:28:37,907 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).
2026-05-18 11:28:38,373 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernic

b55030510eab122a1d5b0ab39709e39b.zip:   0%|          | 0.00/12.8k [00:00<?, ?B/s]

2026-05-18 11:29:02,047 INFO Request ID is 5541a0cb-dffa-4b7f-9651-3c252001c0f1
2026-05-18 11:29:02,126 INFO status has been updated to accepted
2026-05-18 11:29:35,375 INFO status has been updated to successful


5cedeb0463917a12d0afbe87cf657373.zip:   0%|          | 0.00/94.2k [00:00<?, ?B/s]

ERA5 raw files saved to: data/raw/era5
Merged ERA5 CSV saved to: data/interim/era5_all_merged.csv


,time,T2M,D2M,RH,U10,V10,SP,TP,BLH,TCC,CBH,WS10,GP_500,GP_850,T_500,T_850,U_500,U_850,V_500,V_850
0,2026-03-01 00:00:00,283.19373,281.63806,90.061051,-1.254028,-0.572311,100798.670,0.000000e+00,31.963226,0.878021,822.22644,1.378451,55447.753906,15084.210938,250.342636,275.176971,9.574036,-0.010620,14.281281,1.972794
1,2026-03-01 01:00:00,282.90924,281.53995,91.184942,-0.835602,-0.614883,100813.670,0.000000e+00,22.768602,0.906525,821.54565,1.037455,55419.699219,15094.523438,250.103119,275.447815,10.276840,0.261047,11.280548,1.783478
2,2026-03-01 02:00:00,282.85132,281.71823,92.651449,-0.774841,0.444016,100812.016,9.536743e-07,28.508335,0.919250,820.26120,0.893045,55359.222656,15084.656250,249.876694,275.475159,9.741043,0.074005,9.810852,1.249084
3,2026-03-01 03:00:00,282.71730,281.67267,93.200513,-0.716339,0.234665,100841.195,1.907349e-06,39.850513,0.813477,396.53946,0.753797,55338.402344,15095.710938,249.700638,275.300751,9.322769,-0.278305,9.539322,0.779373
4,2026-03-01 04:00:00,282.71070,281.80220,94.062501,-1.095779,0.176224,100856.266,5.245209e-06,47.587524,0.965149,252.46167,1.109859,55323.097656,15091.621094,249.479828,275.048279,9.351929,-0.919876,9.568161,0.440033


## Cell 6 — Merge ARPA station tables
ARPA meteorological station data were manually requested from ARPA Lombardia and provided as CSV tables.

This step merges the downloaded station tables into a unified hourly dataset.

In [ ]:
import src.merge_arpa_data as arpa

importlib.reload(arpa)

ARPA_ROOT = PROJECT_ROOT / "data" / "raw" / "arpa"

ARPA_FILES = sorted(ARPA_ROOT.rglob("*.csv"))

ARPA_OUTPUT_FILE = PROJECT_ROOT / "data" / "interim" / "arpa_merged.csv"

arpa_df = arpa.merge_arpa_tables(
    files=ARPA_FILES,
    output_file=ARPA_OUTPUT_FILE,
    time_column="Data-Ora",
    sensor_column="Id Sensore",
    utc_offset_hours=1,
    missing_value=-999,
)

print("ARPA merged CSV saved to:")
print(ARPA_OUTPUT_FILE.relative_to(PROJECT_ROOT))

display(arpa_df.head())

ARPA merged CSV saved to:
data/interim/arpa_merged.csv


,Data-Ora,temperature_mean,wind_direction_mean,relative_humidity_mean,wind_speed_mean,wind_gust_max
0,2025-12-31 23:00:00,3.3,248,81.3,<NA>,1.0
1,2026-01-01 00:00:00,2.9,300,83.0,0.3,1.1
2,2026-01-01 01:00:00,2.3,342,86.2,0.4,0.9
3,2026-01-01 02:00:00,1.9,4,87.4,<NA>,1.0
4,2026-01-01 03:00:00,1.4,20,89.4,0.4,0.9


## Cell 7 — Merge all datasets
This merges `image_features.csv`, `PM25_MI_hourly.csv`, `era5_all_merged.csv`, and `arpa_merged.csv` by UTC time.

In [ ]:
import src.merge_all_data as merge_all

importlib.reload(merge_all)

MERGED_OUTPUT = PROJECT_ROOT / "data" / "interim" / "merged_dataset.csv"

merged_df = merge_all.merge_all_datasets(
    arpa_file=PROJECT_ROOT / "data" / "interim" / "arpa_merged.csv",
    image_file=PROJECT_ROOT / "data" / "interim" / "image_features.csv",
    pm25_file=PROJECT_ROOT / "data" / "interim" / "PM25_MI_hourly.csv",
    era5_file=PROJECT_ROOT / "data" / "interim" / "era5_all_merged.csv",
    output_file=MERGED_OUTPUT,
)

print("Merged dataset saved to:")
print(MERGED_OUTPUT.relative_to(PROJECT_ROOT))

print(f"\nRows: {len(merged_df)}")
print(f"Columns: {merged_df.shape[1]}")

display(merged_df.head())

Merged dataset saved to:
data/interim/merged_dataset.csv

Rows: 220
Columns: 34


,time,R_roi,G_roi,B_roi,S_mean,B_R_ratio,contrast,image_path,PM25,Unit,...,T_850,U_500,U_850,V_500,V_850,temperature_mean,wind_direction_mean,relative_humidity_mean,wind_speed_mean,wind_gust_max
0,2026-03-01 03:00:00,86.927317,103.467195,122.422105,0.290022,1.408327,19.237125,data/raw/images/20260301-0400.jpg,30.829412,ug.m-3,...,275.30075,9.322769,-0.278305,9.539322,0.779373,11.6,73.0,76.2,0.7,2.2
1,2026-03-01 04:00:00,86.576546,103.945815,123.102785,0.295734,1.421895,14.599862,data/raw/images/20260301-0500.jpg,33.409030,ug.m-3,...,275.04828,9.351929,-0.919876,9.568161,0.440033,11.3,84.0,81.4,1.3,3.1
2,2026-03-01 05:00:00,88.301727,103.318487,121.760818,0.273552,1.378918,13.725783,data/raw/images/20260301-0600.jpg,45.526115,ug.m-3,...,274.86044,9.384903,-1.381363,9.571930,0.111649,10.9,74.0,86.8,1.1,2.6
3,2026-03-01 06:00:00,97.634921,127.311369,157.803475,0.381719,1.616261,17.681084,data/raw/images/20260301-0700.jpg,45.983295,ug.m-3,...,274.77585,8.750961,-1.646408,9.686996,-0.082336,10.7,72.0,89.3,1.6,3.3
4,2026-03-01 07:00:00,112.163654,144.063168,172.775825,0.352873,1.540390,21.162996,data/raw/images/20260301-0800.jpg,47.228165,ug.m-3,...,274.84160,8.000458,-1.764099,10.094360,0.228470,10.3,87.0,93.2,2.2,4.2
